In [3]:
# --- Celda 1: preparación ---
# En Colab, descomenta:
# !git clone https://github.com/USUARIO/resistencia-bacteriana.git
# %cd resistencia-bacteriana
import sys, os, numpy as np, matplotlib.pyplot as plt

# Localiza la raíz del repo (la carpeta que contiene src/), sin importar desde dónde se ejecute
raiz = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, raiz)

from src.EdoLib import euler_vec, euler_positivo, referencia_por_tramos
from src.modelo import get_subpoblaciones, conc_constante, conc_ciclica, metricas
from src.euler_model import load_params

p, sc = load_params("B")      # parámetros desde params.yaml (no hardcodear)
x0 = np.array([p["x0"]["S"], p["x0"]["T"], p["x0"]["R"]])
nombres = ["Sensibles", "Tolerantes", "Resistentes"]

def graficar(t, X, titulo):
    for i, n in enumerate(nombres):
        pl = plt.semilogy(t, np.maximum(X[:, i], 0.1), label=n)
    plt.xlabel("Tiempo (días)"); plt.ylabel("Densidad (UFC/mL)")
    plt.title(titulo); plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
# --- Celda 2: Escenario A (exposición constante C=1, 5 días, dt=0.1 d) ---
XA, tA = euler_vec(0, 5, x0, get_subpoblaciones, 0.1, p, conc_constante(1.0))
graficar(tA, XA, "Escenario A: exposición constante")
print("Estado final A:", XA[-1])

In [ ]:
# --- Celda 2: Escenario A (exposición constante C=1, 5 días, dt=0.1 d) ---
XA, tA = euler_vec(0, 5, x0, get_subpoblaciones, 0.1, p, conc_constante(1.0))
graficar(tA, XA, "Escenario A: exposición constante")
print("Estado final A:", XA[-1])

In [ ]:
# --- Celda 3: Escenario B (3 h con antibiótico C=5, 21 h sin; dt=0.5 h) ---
dt = 0.5 / 24
XB, tB = euler_vec(0, 20, x0, get_subpoblaciones, dt, p, conc_ciclica())
graficar(tB, XB, "Escenario B: ciclos de 3 h")
print(metricas(tB, XB))

In [ ]:
# --- Celda 4: validación de no negatividad ---
XP, _ = euler_positivo(0, 20, x0, get_subpoblaciones, dt, p, conc_ciclica())
print("mínimo de B:", XB.min())
print("¿clipping cambió algo?:", not np.allclose(XB, XP))

In [ ]:
# --- Celda 5: validación contra la referencia (¿cuánto error tiene Euler?) ---
ref, _ = referencia_por_tramos(20, x0, get_subpoblaciones, 0.05/24, p)
print("dt (h)     err_S      err_T      err_R")
for h in [1.0, 0.5, 0.25, 0.125, 0.0625]:
    X, _ = euler_vec(0, 20, x0, get_subpoblaciones, h/24, p, conc_ciclica())
    e = np.abs(X[-1] - ref[-1]) / ref[-1]
    print(f"{h:6.4f}   {e[0]:.1e}   {e[1]:.1e}   {e[2]:.1e}")